<a href="https://colab.research.google.com/github/Ziyi-star/Bachelorarbeit/blob/main/notebooks/training/train_1s_30hz_initial_2class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Field Evaluation Anayse 1s 100hz

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pickle
import scipy
import datetime
import numpy as np
import tensorflow as tf
import sys
import matplotlib.pyplot as plt
sys.path.append('../../')
from utils.preprocessing import *

## 3. Analyses

In [3]:
df_fn = pd.read_csv('../../data/Field_validation/false_negatives_100hz.csv')
df_fn.head()

,Index,True Label,Predicted Label,Prediction Score (Class 0),Prediction Score (Class 1),Start Time,End Time
0,273,1.0,0,0.988674,1.132586e-02,2025-10-21 15:20:13.350,2025-10-21 15:20:14.340
1,283,1.0,0,0.825433,1.745667e-01,2025-10-21 15:20:23.350,2025-10-21 15:20:24.340
2,287,1.0,0,0.976598,2.340220e-02,2025-10-21 15:20:27.350,2025-10-21 15:20:28.340
3,288,1.0,0,1.000000,3.727133e-07,2025-10-21 15:20:28.350,2025-10-21 15:20:29.340
4,289,1.0,0,0.999984,1.599910e-05,2025-10-21 15:20:29.350,2025-10-21 15:20:30.340


In [4]:
df_video = pd.read_csv('../../data/Field_validation/video.csv')
df_video.head()

,NTP,Timestamp,Video,X,Y,Z
0,"2025-10-21, 15:15:42.0410","2025-10-21, 15:15:42.0390",00:00:00.104,-0.161850,-0.621307,-0.760971
1,"2025-10-21, 15:15:42.0420","2025-10-21, 15:15:42.0410",00:00:00.105,-0.166687,-0.623611,-0.760941
2,"2025-10-21, 15:15:42.0490","2025-10-21, 15:15:42.0480",00:00:00.112,-0.161774,-0.620071,-0.765106
3,"2025-10-21, 15:15:42.0520","2025-10-21, 15:15:42.0500",00:00:00.114,-0.156738,-0.621979,-0.765808
4,"2025-10-21, 15:15:42.0530","2025-10-21, 15:15:42.0520",00:00:00.116,-0.152634,-0.622070,-0.766769


### a)Get video positions for each false negative

In [8]:
buffer = 0.1  # seconds
td = pd.Timedelta(seconds=buffer)

# Ensure datetime types
df_fn['Start Time'] = pd.to_datetime(df_fn['Start Time'], errors='coerce')
df_video['NTP'] = pd.to_datetime(df_video['NTP'], errors='coerce')

df_fn['video'] = np.nan
df_fn['matched_ntp'] = np.nan

for idx, row in df_fn.iterrows():
    start_time = row['Start Time']

    candidates = df_video[
        (df_video['NTP'] >= start_time - td) &
        (df_video['NTP'] <= start_time + td)
    ]

    if len(candidates):
        # Work on a copy to avoid chained assignment warnings
        candidates = candidates.copy()
        # Compute absolute time difference to the FN start time
        candidates['time_diff'] = (candidates['NTP'] - start_time).abs()
        # Pick the candidate with the smallest time difference
        best = candidates.loc[candidates['time_diff'].idxmin()]

        # Write matched info back to df_fn
        df_fn.at[idx, 'video'] = best['Video']   # matched video name/id
        df_fn.at[idx, 'matched_ntp'] = best['NTP']           # the matched NTP timestamp
        df_fn.at[idx, 'time_diff'] = best['time_diff']       # absolute time difference



print(df_fn[['Index', 'Start Time', 'video', 'matched_ntp', 'time_diff']])

    Index              Start Time         video                 matched_ntp  \
0     273 2025-10-21 15:20:13.350  00:04:31.412  2025-10-21 15:20:13.350000   
1     283 2025-10-21 15:20:23.350  00:04:41.408  2025-10-21 15:20:23.346000   
2     287 2025-10-21 15:20:27.350  00:04:45.415  2025-10-21 15:20:27.352000   
3     288 2025-10-21 15:20:28.350  00:04:46.411  2025-10-21 15:20:28.348000   
4     289 2025-10-21 15:20:29.350  00:04:47.408  2025-10-21 15:20:29.345000   
5     293 2025-10-21 15:20:33.350  00:04:51.414  2025-10-21 15:20:33.351000   
6     295 2025-10-21 15:20:35.350  00:04:53.417  2025-10-21 15:20:35.355000   
7     306 2025-10-21 15:20:46.350  00:05:04.409  2025-10-21 15:20:46.347000   
8     617 2025-10-21 15:25:57.350  00:10:15.411  2025-10-21 15:25:57.348000   
9     618 2025-10-21 15:25:58.350  00:10:16.407  2025-10-21 15:25:58.345000   
10    751 2025-10-21 15:28:11.350  00:12:29.415  2025-10-21 15:28:11.353000   
11    752 2025-10-21 15:28:12.350  00:12:30.412  202

C:\Users\liuzi\AppData\Local\Temp\ipykernel_10860\4120437995.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '00:04:31.412' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_fn.at[idx, 'video'] = best['Video']   # matched video name/id
C:\Users\liuzi\AppData\Local\Temp\ipykernel_10860\4120437995.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2025-10-21 15:20:13.350000' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_fn.at[idx, 'matched_ntp'] = best['NTP']           # the matched NTP timestamp
